# Exploratory Data Analysis: Federal Procurement Data

**Dissertation:** From Lowest Price to Highest Public Value

This notebook provides an initial exploration of the federal procurement data
downloaded from USAspending.gov. The goal is to understand the structure,
coverage, and quality of the data before formal analysis.

## Key Questions
1. How are awards distributed between LPTA and best-value tradeoff?
2. What is the breakdown by agency, NAICS code, and fiscal year?
3. What does the competition landscape look like (number of offers)?
4. What is the distribution of contract values?
5. What data quality issues exist (missing values, coding inconsistencies)?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path

# Set display options
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.2f}'.format)

# Project paths
PROJECT_ROOT = Path.home() / 'phd-research'
RAW_DATA = PROJECT_ROOT / 'data' / 'raw' / 'usaspending'
PROCESSED_DATA = PROJECT_ROOT / 'data' / 'processed'

print(f'Project root: {PROJECT_ROOT}')
print(f'Raw data dir: {RAW_DATA}')
print(f'Looking for data files...')

# List available data files
if RAW_DATA.exists():
    for f in sorted(RAW_DATA.rglob('*.csv')):
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f'  {f.relative_to(RAW_DATA)} ({size_mb:.1f} MB)')
else:
    print('  No data downloaded yet. Run: python data/usaspending/download_awards.py --fy 2020-2025')

## 1. Load and Inspect Data

Load the USAspending contract data. Key columns of interest:
- `type_of_set_aside` - competition type
- `extent_competed` - competition extent
- `evaluated_preference` - source selection approach
- `type_of_contract_pricing` - contract type (FFP, CPFF, etc.)
- `naics_code` - industry classification
- `product_or_service_code` - PSC
- `total_obligated_amount` - dollars obligated
- `number_of_offers_received` - competition level

In [ ]:
# Load data - adjust the filename based on what was downloaded
# USAspending bulk downloads come as zipped CSVs
import zipfile
import glob

dfs = []
zip_files = sorted(RAW_DATA.rglob('*.zip'))

if not zip_files:
    print('No ZIP files found. Checking for CSVs directly...')
    csv_files = sorted(RAW_DATA.rglob('*.csv'))
    for csv_file in csv_files[:3]:  # Load first 3 to start
        print(f'Loading {csv_file.name}...')
        df = pd.read_csv(csv_file, low_memory=False)
        dfs.append(df)
else:
    for zf in zip_files:
        print(f'Extracting {zf.name}...')
        with zipfile.ZipFile(zf, 'r') as z:
            for name in z.namelist():
                if name.endswith('.csv') and 'contract' in name.lower():
                    print(f'  Loading {name}...')
                    with z.open(name) as f:
                        df = pd.read_csv(f, low_memory=False)
                        dfs.append(df)

if dfs:
    data = pd.concat(dfs, ignore_index=True)
    print(f'\nTotal records loaded: {len(data):,}')
    print(f'Columns: {len(data.columns)}')
    print(f'\nFirst few column names:')
    for col in data.columns[:30]:
        print(f'  {col}')
else:
    print('\nNo data files found yet.')
    print('Run this command first:')
    print('  python ~/phd-research/data/usaspending/download_awards.py --fy 2020-2025')
    data = pd.DataFrame()  # empty placeholder

In [ ]:
# Basic data overview
if not data.empty:
    print('=== Data Shape ===')
    print(f'Rows: {len(data):,}')
    print(f'Columns: {len(data.columns)}')
    print()
    print('=== Data Types ===')
    print(data.dtypes.value_counts())
    print()
    print('=== Missing Values (top 20) ===')
    missing = data.isnull().sum().sort_values(ascending=False)
    missing_pct = (missing / len(data) * 100).round(1)
    print(pd.DataFrame({'missing': missing[:20], 'pct': missing_pct[:20]}))
else:
    print('No data loaded yet.')

## 2. Source Selection Process Analysis

The critical variable: how awards are distributed between:
- **LPTA** (Lowest Price Technically Acceptable)
- **Best-Value Tradeoff** (price vs. non-price factors)
- **Other** methods

In FPDS, the `evaluated_preference` or source selection process field encodes this.

In [ ]:
# Identify the source selection column
# USAspending uses different column names depending on the download type
if not data.empty:
    # Look for source selection related columns
    ss_candidates = [col for col in data.columns if any(term in col.lower() 
                     for term in ['evaluated', 'source_selection', 'selection_process',
                                  'type_of_set_aside', 'extent_competed'])]
    print('Source selection related columns found:')
    for col in ss_candidates:
        print(f'\n  {col}:')
        print(f'  {data[col].value_counts().head(10).to_string()}')
else:
    print('No data loaded yet.')

In [ ]:
# Distribution of awards by source selection method
if not data.empty:
    # Adjust column name based on what's found above
    # Common names: 'evaluated_preference', 'fair_opportunity_limited_sources'
    
    # Competition analysis
    competition_cols = [col for col in data.columns if 'compet' in col.lower()]
    for col in competition_cols:
        print(f'\n{col}:')
        vc = data[col].value_counts()
        print(vc.head(15))
        
        # Visualize
        fig, ax = plt.subplots(figsize=(10, 5))
        vc.head(10).plot(kind='barh', ax=ax)
        ax.set_title(f'Distribution: {col}')
        ax.set_xlabel('Count')
        plt.tight_layout()
        plt.show()
else:
    print('No data loaded yet.')

## 3. Agency and Category Breakdown

In [ ]:
if not data.empty:
    # Top agencies by award count
    agency_col = [col for col in data.columns if 'agency' in col.lower() and 'name' in col.lower()]
    if agency_col:
        col = agency_col[0]
        print(f'Top 20 Agencies by Award Count ({col}):')
        print(data[col].value_counts().head(20))
        
        fig, ax = plt.subplots(figsize=(12, 8))
        data[col].value_counts().head(15).plot(kind='barh', ax=ax)
        ax.set_title('Top 15 Agencies by Award Count')
        ax.set_xlabel('Number of Awards')
        plt.tight_layout()
        plt.show()
    
    # NAICS code distribution
    naics_col = [col for col in data.columns if 'naics' in col.lower()]
    if naics_col:
        col = naics_col[0]
        print(f'\nTop 20 NAICS Codes ({col}):')
        print(data[col].value_counts().head(20))
else:
    print('No data loaded yet.')

## 4. Contract Value Distribution

In [ ]:
if not data.empty:
    # Find the obligation/amount column
    amount_cols = [col for col in data.columns if any(term in col.lower() 
                   for term in ['obligat', 'amount', 'award_amount', 'total_dollars'])]
    print('Amount columns found:', amount_cols)
    
    if amount_cols:
        col = amount_cols[0]
        amounts = pd.to_numeric(data[col], errors='coerce').dropna()
        
        print(f'\n=== {col} Statistics ===')
        print(f'Count: {len(amounts):,}')
        print(f'Mean: ${amounts.mean():,.0f}')
        print(f'Median: ${amounts.median():,.0f}')
        print(f'Std Dev: ${amounts.std():,.0f}')
        print(f'Min: ${amounts.min():,.0f}')
        print(f'Max: ${amounts.max():,.0f}')
        print(f'Total: ${amounts.sum():,.0f}')
        
        # Distribution plot (log scale for readability)
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Histogram
        positive = amounts[amounts > 0]
        axes[0].hist(np.log10(positive), bins=50, edgecolor='black', alpha=0.7)
        axes[0].set_title('Distribution of Award Amounts (log10)')
        axes[0].set_xlabel('log10(Amount)')
        axes[0].set_ylabel('Count')
        
        # Box plot by quintile
        axes[1].boxplot([positive], vert=True)
        axes[1].set_title('Award Amount Box Plot')
        axes[1].set_ylabel('Amount ($)')
        axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
        
        plt.tight_layout()
        plt.show()
else:
    print('No data loaded yet.')

## 5. Competition Analysis

In [ ]:
if not data.empty:
    # Number of offers received
    offers_col = [col for col in data.columns if 'offer' in col.lower() or 'bid' in col.lower()]
    print('Offers/bid columns:', offers_col)
    
    if offers_col:
        col = offers_col[0]
        offers = pd.to_numeric(data[col], errors='coerce').dropna()
        
        print(f'\n=== Number of Offers ({col}) ===')
        print(f'Mean: {offers.mean():.1f}')
        print(f'Median: {offers.median():.0f}')
        print(f'% Single bid: {(offers == 1).mean()*100:.1f}%')
        print(f'% 2-3 bids: {((offers >= 2) & (offers <= 3)).mean()*100:.1f}%')
        print(f'% 4+ bids: {(offers >= 4).mean()*100:.1f}%')
        
        fig, ax = plt.subplots(figsize=(10, 5))
        offers.clip(upper=20).hist(bins=20, ax=ax, edgecolor='black', alpha=0.7)
        ax.set_title('Distribution of Number of Offers Received')
        ax.set_xlabel('Number of Offers')
        ax.set_ylabel('Count')
        plt.tight_layout()
        plt.show()
else:
    print('No data loaded yet.')

## 6. Article Database Summary

Quick look at the scholarly articles collected by the daily search automation.

In [ ]:
import sqlite3

articles_db = PROJECT_ROOT / 'research' / 'articles' / 'articles.db'
if articles_db.exists():
    conn = sqlite3.connect(str(articles_db))
    
    # Total articles
    total = pd.read_sql('SELECT COUNT(*) as count FROM articles', conn).iloc[0, 0]
    print(f'Total articles in database: {total}')
    
    # Top articles by citation count (procurement-relevant)
    top_articles = pd.read_sql('''
        SELECT title, authors, year, journal, citation_count 
        FROM articles 
        WHERE title LIKE '%procurement%' 
           OR title LIKE '%bid protest%' 
           OR title LIKE '%source selection%'
           OR title LIKE '%LPTA%'
           OR title LIKE '%best value%'
           OR title LIKE '%public value%'
           OR title LIKE '%auction%'
           OR title LIKE '%transaction cost%'
        ORDER BY citation_count DESC 
        LIMIT 20
    ''', conn)
    
    print(f'\nTop 20 Procurement-Relevant Articles:')
    for _, row in top_articles.iterrows():
        print(f"  [{row['citation_count']:>5} cites] {row['title'][:70]}")
        print(f"             {row['authors'][:50]} ({row['year']})")
    
    # Articles by year
    by_year = pd.read_sql('SELECT year, COUNT(*) as count FROM articles WHERE year IS NOT NULL GROUP BY year ORDER BY year', conn)
    
    fig, ax = plt.subplots(figsize=(12, 5))
    by_year.plot(x='year', y='count', kind='bar', ax=ax, legend=False)
    ax.set_title('Articles in Database by Publication Year')
    ax.set_xlabel('Year')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()
    
    conn.close()
else:
    print('Article database not found. Run the daily search first.')

## Next Steps

After this initial exploration:

1. **Data Cleaning** - Handle missing values, standardize codes, filter to study sample
2. **Variable Construction** - Build the source selection indicator, public value index
3. **Descriptive Statistics** - Full summary tables for the dissertation
4. **Sample Selection** - Apply inclusion/exclusion criteria per the proposal
5. **Propensity Score Estimation** - Model method choice (LPTA vs tradeoff)
6. **Outcome Analysis** - DiD and matched comparisons